# Atelier Scikit-learn — Prédiction de l'état d'un capteur IoT

**Objectif :** construire un modèle de Machine Learning capable de prédire automatiquement l'état d'un capteur (`OK`, `ALERTE`, `ERREUR`) à partir de ses mesures (température, humidité, pression, consommation).

**Workflow suivi :**
`Dataset → Chargement → Exploration → Nettoyage → X/y → Train/Test → Prétraitement → Modèle → fit() → predict() → Évaluation → Sauvegarde → Chargement → Réutilisation`

Structure du projet :
```
SmartSensor_SKL_ML/
├── data/mesures_capteurs.csv
├── notebooks/SmartSensor_SKL_ML.ipynb
└── models/modele_capteurs.joblib, modele_capteurs.pkl
```


### Installation, imports et chargement des données

On importe Pandas (données), Seaborn/Matplotlib (visualisation de l'évaluation), et les modules Scikit-learn nécessaires à chaque étape du pipeline.

In [3]:
# Installation si nécessaire
!pip install scikit-learn seaborn matplotlib pandas joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib
import pickle

sns.set_theme(style="whitegrid")

# Import du dataset
df = pd.read_csv("../data/mesures_capteurs.csv")
df.head()


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


In [4]:
# 6) Exploration du dataframe
print("Dimensions :", df.shape)
df.info()


Dimensions : (605, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    object 
 1   date_heure    605 non-null    object 
 2   id_capteur    605 non-null    object 
 3   batiment      605 non-null    object 
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    object 
dtypes: float64(4), object(5)
memory usage: 42.7+ KB


## Partie 1 : Gestion des doublons

Des mesures dupliquées fausseraient l'apprentissage du modèle (le même exemple compterait plusieurs fois) : elles doivent être supprimées avant toute étape de modélisation.

In [5]:
# 1) Vérification de l'existence de doublons
print("Nombre de doublons :", df.duplicated().sum())

# 2) Suppression puis vérification
df = df.drop_duplicates()
print("Nombre de doublons après suppression :", df.duplicated().sum())
print("Nouvelles dimensions :", df.shape)


Nombre de doublons : 5
Nombre de doublons après suppression : 0
Nouvelles dimensions : (600, 9)


## Partie 2 : Sélection de y (cible) et X (caractéristiques)

`y` est la variable à prédire (`etat`), `X` regroupe les variables explicatives utilisées pour faire cette prédiction. Une ligne dont la cible est manquante ne peut pas servir à l'apprentissage : on la retire avant de séparer X et y.

In [6]:
# On retire les lignes dont l'état (cible) est manquant : impossible d'apprendre sans cible connue
df_modele = df.dropna(subset=["etat"])
print("Lignes avec état manquant retirées :", df.shape[0] - df_modele.shape[0])

# 1) Définition de X (caractéristiques) et y (cible)
X = df_modele[["temperature", "humidite", "pression", "consommation"]]
y = df_modele["etat"]


Lignes avec état manquant retirées : 4


In [7]:
# 2) Cinq premières lignes de X et de y
print("X :")
display(X.head())
print("y :")
display(y.head())


X :


,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


y :


0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: object

**3) Type de problème :** c'est un problème de **classification multi-classe supervisée** : on cherche à prédire une catégorie parmi trois (`OK`, `ALERTE`, `ERREUR`) à partir de variables numériques.